In [39]:
import math
import torch
from einops import einsum

# Test Linear

In [2]:
from modules import LinearLayer

linear1 = LinearLayer(in_features=4, out_features=3)
linear1.W.shape

torch.Size([3, 4])

In [3]:
x1d = torch.randn(4)           # (d_in)
x2d = torch.randn(5, 4)        # (batch, d_in)
x3d = torch.randn(2, 5, 4)     # (batch, seq, d_in)

print(linear1(x1d).shape)
print(linear1(x2d).shape)
print(linear1(x3d).shape)

torch.Size([3])
torch.Size([5, 3])
torch.Size([2, 5, 3])


# Test Embedding

In [4]:
from modules import EmbeddingLayer

emb = EmbeddingLayer(10, 4)  # vocab=10, d_model=4

x0 = torch.tensor(3)                     # scalar
x1 = torch.tensor([1, 2, 3])              # (seq_len,)
x2 = torch.tensor([[1, 2], [3, 4]])       # (batch, seq_len)

print(emb(x0).shape, emb(x0).grad_fn)
print(emb(x1).shape, emb(x1).grad_fn)
print(emb(x2).shape, emb(x2).grad_fn)

torch.Size([4]) <SelectBackward0 object at 0x10942b8b0>
torch.Size([3, 4]) <IndexBackward0 object at 0x11fd10f70>
torch.Size([2, 2, 4]) <IndexBackward0 object at 0x10942b8b0>


In [5]:
emb(x2).grad_fn

# Test RMSNorm

In [6]:
from modules import RMSNormLayer

norm = RMSNormLayer(4)
x = torch.randn(2, 3, 4)  # (batch, seq_len, d_model)
out = norm(x)

print(out.shape)  # should be (2, 3, 4) — same as input
print(out.dtype)  # should match x.dtype
print(x.dtype)

torch.Size([2, 3, 4])
torch.float32
torch.float32


# Test SwiGLU

In [7]:
from modules import SwiGLU

swiglu = SwiGLU(d_model=8, d_ff=24)
x = torch.randn(2, 5, 8)
out = swiglu(x)

print(out.shape)  # what do you expect, and what do you get?

torch.Size([2, 5, 8])


In [8]:
buf = torch.randn(10, 4)  # (max_seq_len, d_k//2)
pos = torch.tensor([0, 1, 2])  # (seq_len,)
print(buf[pos].shape)  # what do you get?

pos_batched = torch.tensor([[0,1,2],[3,4,5]])  # (batch, seq_len)
print(buf[pos_batched].shape)  # what do you get?

torch.Size([3, 4])
torch.Size([2, 3, 4])


# Test RoPE

In [9]:
from modules import RoPE

# Test Softmax

In [11]:
from modules import SoftmaxLayer

x = torch.randn(size=(2, 2, 4))
SoftmaxLayer(x, dim=2)

tensor([[[0.6459, 0.0547, 0.1019, 0.1974],
         [0.3798, 0.2787, 0.2760, 0.0654]],

        [[0.3412, 0.0291, 0.2453, 0.3844],
         [0.4448, 0.2717, 0.0826, 0.2009]]])

In [47]:
Q = torch.randn(2, 3, 2)
K = torch.randn(2, 4, 2)
V = torch.randn(4,2)
d_k = 2
mask = torch.randint(high=2, size=(3,4))
mask = torch.where(mask == 1, True, False)
mask

tensor([[ True, False, False,  True],
        [ True, False, False,  True],
        [ True,  True,  True, False]])

In [52]:
Q_K_T = einsum(Q, K, " ... n d_k, ... m d_k -> ... n m") / math.sqrt(d_k)
masked_Q_K_T = torch.where(mask, Q_K_T, float('-inf'))
probs = SoftmaxLayer(masked_Q_K_T, dim=-1)
attention_value = einsum(probs, V, " ... n m, ... m d_v -> ... n d_v")
attention_value

tensor([[[-0.9091, -1.3137],
         [ 0.0196, -0.3278],
         [-0.0581, -0.8864]],

        [[-0.4017, -0.7750],
         [-0.3137, -0.6817],
         [ 0.5179, -2.4577]]])